In [ ]:
%pip install openai

  Using cached anyio-4.12.1-py3-none-any.whl.metadata (4.3 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached jiter-0.13.0-cp314-cp314-win_amd64.whl.metadata (5.3 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached certifi-2026.2.25-py3-none-any.whl.metadata (2.5 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.41.5-cp314-cp314-win_amd64.whl.metadata (7.4 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
from openai import OpenAI


def _read_hf_token():
    # Try environment variables first.
    token = (
        os.getenv("HF_TOKEN")
    )
    if token:
        return token.strip().strip('"').strip("'")

    # Fallback: parse local .env in notebook working directory.
    try:
        with open(".env", "r", encoding="utf-8") as env_file:
            for raw_line in env_file:
                line = raw_line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                key, value = line.split("=", 1)
                key = key.strip()
                value = value.strip().strip('"').strip("'")
                if key in {"HF_TOKEN", "HugginFaceToken", "HuggingFaceToken"}:
                    return value
    except FileNotFoundError:
        pass

    return None


hf_token = _read_hf_token()
if not hf_token:
    raise ValueError(
        "Hugging Face token not found. Set HF_TOKEN or HugginFaceToken in .env."
    )

client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=hf_token,
)

ListOfModels = [
    "zai-org/GLM-5:novita",
    "moonshotai/Kimi-K2.5:novita",
    "Qwen/Qwen3.5-397B-A17B:novita",
    "MiniMaxAI/MiniMax-M2.5:novita",
    "deepseek-ai/DeepSeek-V3.2:novita",
    "XiaomiMiMo/MiMo-V2-Flash:novita",
    "openai/gpt-oss-120b:groq",
    "meta-llama/Llama-4-Maverick-17B-128E-Instruct:groq",
    "stepfun-ai/Step-3.5-Flash:featherless-ai",
    "deepseek-ai/DeepSeek-R1:novita"
]

instruction = """
# ROLE
You are a professional children's storyteller writing for an animated YouTube channel. Your task is to write a funny, educational story about a lunar eclipse, inspired by the energetic and curiosity-driven style of Korean educational comics like "Why?".

# CRITICAL FORMATTING CONSTRAINTS
- Output ONLY plain text story narration.
- DO NOT include ANY HTML or XML tags (for example: <p>, <div>, <br>, <!DOCTYPE html>, <html>, <body>).
- DO NOT include Markdown formatting (for example: headings, bullet lists, code blocks, or backticks).
- The response must be raw text only, as if it were read aloud directly.
- DO NOT include: Stage directions, panel descriptions, animation cues, formatting instructions, brackets, production notes, or character names with colons (e.g., "Narrator:").
- Target Length: 800-950 words (approximately 5 minutes when read aloud).
- Pacing: Use a natural spoken storytelling rhythm with lively, dynamic sentences. Avoid overly long paragraphs.

# TARGET AUDIENCE & TONE
- Audience: Children aged 8-12 years old who are curious, easily distracted, love dramatic reactions, and constantly ask "Why?".
- Tone: Funny, playfully dramatic, curious, warm, and fast-paced.
- Style Rules:
  - Make it feel like an exciting discovery.
  - Include exaggerated reactions and funny misunderstandings.
  - Naturally integrate science explanations into the dialogue and events.
  - Include at least three (3) exaggerated "WHY?!" moments.
  - Keep humor playful and silly (strictly no sarcasm or dark humor).

# EDUCATIONAL CONTENT REQUIREMENTS
You must clearly and accurately explain:
1. What a lunar eclipse is.
2. How Earth blocks sunlight from reaching the Moon.
3. Why lunar eclipses don't happen every month.
4. The difference between a lunar and solar eclipse.
5. That it is completely safe to look at a lunar eclipse.

*Explanation Guidelines:* Use simple, highly visual comparisons. Avoid complex physics jargon, textbook-style lectures, and horror tones.

Now, generate the full 5-minute storytelling script following all instructions above.
"""

print(f"Total models loaded: {len(ListOfModels)}")

Total models loaded: 10


In [8]:
import time
import re
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FuturesTimeoutError


def compare_models(instruction_text, model_list, max_retries=3, request_timeout=180):
    words_per_minute = 150
    words_per_second = words_per_minute / 60
    results = []

    # Global output guardrail so every prompt requests plain text only.
    plain_text_guardrail = (
        "\n\nOUTPUT FORMAT REQUIREMENTS (STRICT):\n"
        "- Return plain text only.\n"
        "- Do not return HTML, XML, or any tags like <p>, <div>, <br>, <!DOCTYPE html>.\n"
        "- Do not return Markdown, code fences, or backticks.\n"
        "- Return only the final story text.\n"
    )

    # Hard timeout to prevent indefinite hangs (slightly longer than request_timeout).
    hard_timeout_seconds = request_timeout + 30

    def _call_api():
        return client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": instruction_text + plain_text_guardrail}],
            temperature=0.7,
            timeout=request_timeout,
        )

    print("=" * 80)

    for index, model_name in enumerate(model_list):
        print(f"[{index + 1}/{len(model_list)}] Generating with: {model_name}...\n")

        completion = None
        start_time = time.time()

        for attempt in range(1, max_retries + 1):
            try:
                with ThreadPoolExecutor(max_workers=1) as executor:
                    future = executor.submit(_call_api)
                    completion = future.result(timeout=hard_timeout_seconds)
                break
            except FuturesTimeoutError:
                error_msg = f"Request timed out after {hard_timeout_seconds}s (API call did not return)"
            except Exception as err:
                error_msg = str(err)
            else:
                continue  # success: break exits loop; else is unreachable but required for fall-through
            is_last_attempt = attempt == max_retries
            wait_seconds = min(2 ** attempt, 10)
            # Detect router/gateway errors (HTML responses, timeouts, etc.)
            if "<!DOCTYPE html>" in error_msg or "504" in error_msg or "Gateway" in error_msg:
                error_type = "API Gateway Error (likely service issue)"
            else:
                error_type = "Model/API Error"
            print(f"Attempt {attempt}/{max_retries} failed for {model_name}: {error_type}")
            print(f"Error detail: {error_msg}")
            if is_last_attempt:
                api_duration = time.time() - start_time
                results.append(
                    {
                        "model": model_name,
                        "instruction_text": instruction_text,
                        "api_duration": api_duration,
                        "story_text": "",
                        "word_count": 0,
                        "estimated_tts_seconds": 0,
                        "paragraph_count": 0,
                        "avg_sentence_length": 0,
                        "is_in_target_length": False,
                        "error": error_type,
                        "error_detail": error_msg,
                    }
                )
                print(f"Skipping model after {max_retries} failed attempts: {model_name}")
                print("-" * 80)
            else:
                print(f"Retrying in {wait_seconds}s...")
                time.sleep(wait_seconds)

        if completion is None:
            continue

        api_duration = time.time() - start_time
        story_text = completion.choices[0].message.content or ""
        words = story_text.split()
        word_count = len(words)

        estimated_tts_seconds = word_count / words_per_second if word_count > 0 else 0
        tts_minutes = int(estimated_tts_seconds // 60)
        tts_seconds = int(estimated_tts_seconds % 60)

        paragraphs = [paragraph for paragraph in story_text.split("\n") if paragraph.strip()]
        paragraph_count = len(paragraphs)
        sentences = [sentence for sentence in re.split(r"[.!?]+", story_text) if sentence.strip()]
        avg_sentence_length = word_count / max(1, len(sentences))

        is_in_target_length = 800 <= word_count <= 950

        result = {
            "model": model_name,
            "instruction_text": instruction_text,
            "api_duration": api_duration,
            "story_text": story_text,
            "word_count": word_count,
            "estimated_tts_seconds": estimated_tts_seconds,
            "paragraph_count": paragraph_count,
            "avg_sentence_length": avg_sentence_length,
            "is_in_target_length": is_in_target_length,
            "error": None,
            "error_detail": None,
        }
        results.append(result)

        print("MODEL ANALYSIS:")
        print(f"API Response Time: {api_duration:.2f} seconds")
        print(f"Word Count: {word_count} words (Target: 800-950) -> {'Pass' if is_in_target_length else 'Fail'}")
        print(f"Est. TTS Duration: {tts_minutes}m {tts_seconds}s")
        print(f"Paragraph Count: {paragraph_count}")
        print(f"Pacing: ~{avg_sentence_length:.1f} words per sentence")
        print("-" * 80)

        print("FULL STORY OUTPUT:\n")
        print(story_text)
        print("\n" + "=" * 80 + "\n")

    return results


def compare_instruction_sets(instruction_texts, model_list):
    if isinstance(instruction_texts, str):
        instruction_texts = [instruction_texts]

    all_results = []

    for instruction_index, instruction_text in enumerate(instruction_texts, start=1):
        print(f"INSTRUCTION [{instruction_index}/{len(instruction_texts)}]")
        instruction_results = compare_models(instruction_text, model_list)
        all_results.append(
            {
                "instruction_index": instruction_index,
                "instruction_text": instruction_text,
                "results": instruction_results,
            }
        )

    return all_results

In [ ]:
instruction_list = [instruction]
comparison_results = compare_models(instruction, ListOfModels)

[1/10] Generating with: zai-org/GLM-5:novita...

MODEL ANALYSIS:
API Response Time: 44.11 seconds
Word Count: 1147 words (Target: 800-950) -> Fail
Est. TTS Duration: 7m 38s
Paragraph Count: 38
Pacing: ~8.0 words per sentence
--------------------------------------------------------------------------------
FULL STORY OUTPUT:

The night started like any other night in the little town of Stargazer Valley. Twelve-year-old Milo was doing what he always did before bed. He was staring out his window at the Moon, making up stories about the rabbits and dragons that supposedly lived up there. His older sister, Luna, kept telling him those stories were nonsense, but Milo didn't care. The Moon was his friend.

But tonight, something was wrong. Very, very wrong.

Milo squinted at the bright white circle in the sky. A dark shadow was creeping across it, slowly eating it alive! His eyes went wide. His jaw dropped. He grabbed his telescope and looked again. The shadow was real, and it was getting bigg

In [ ]:
EducationMaterial = """
Solids
A solid’s particles are packed closely together. The forces between the particles are strong enough that the particles cannot move freely; they can only vibrate. As a result, a solid has a stable, definite shape and a definite volume. Solids can only change shape under force, as when broken or cut.

In crystalline solids, particles are packed in a regularly ordered, repeating pattern. There are many different crystal structures, and the same substance can have more than one structure. For example, iron has a body-centered cubic structure at temperatures below 912 °C and a face-centered cubic structure between 912 and 1,394 °C. Ice has 15 known crystal structures, each of which exists at a different temperature and pressure.

A solid can transform into a liquid through melting, and a liquid can transform into a solid through freezing. A solid can also change directly into a gas through a process called sublimation.

Liquids
A liquid is a fluid that conforms to the shape of its container but that retains a nearly constant volume independent of pressure. The volume is definite (does not change) if the temperature and pressure are constant. When a solid is heated above its melting point, it becomes liquid because the pressure is higher than the triple point of the substance. Intermolecular (or interatomic or interionic) forces are still important, but the molecules have enough energy to move around, which makes the structure mobile. This means that a liquid is not definite in shape but rather conforms to the shape of its container. Its volume is usually greater than that of its corresponding solid (water is a well-known exception to this rule). The highest temperature at which a particular liquid can exist is called its critical temperature.

A liquid can be converted to a gas through heating at constant pressure to the substance’s boiling point or through reduction of pressure at constant temperature. This process of a liquid changing to a gas is called evaporation.

Gases
Gas molecules have either very weak bonds or no bonds at all, so they can move freely and quickly. Because of this, not only will a gas conform to the shape of its container, it will also expand to completely fill the container. Gas molecules have enough kinetic energy that the effect of intermolecular forces is small (or zero, for an ideal gas), and they are spaced very far apart from each other; the typical distance between neighboring molecules is much greater than the size of the molecules themselves.

A gas at a temperature below its critical temperature can also be called a vapor. A vapor can be liquefied through compression without cooling. It can also exist in equilibrium with a liquid (or solid), in which case the gas pressure equals the vapor pressure of the liquid (or solid).

A supercritical fluid (SCF) is a gas whose temperature and pressure are greater than the critical temperature and critical pressure. In this state, the distinction between liquid and gas disappears. A supercritical fluid has the physical properties of a gas, but its high density lends it the properties of a solvent in some cases. This can be useful in several applications. For example, supercritical carbon dioxide is used to extract caffeine in the manufacturing of decaffeinated coffee.
"""

story_prompt_templates = [
    """You are an expert children's author specializing in the "Korean Why? comic" style. Your task is to write a highly engaging, slapstick, and educational 2,000-word story for children aged 8-12.

Characters: Comji (impulsive/funny), Omji (smart/logical), and Professor Quark (eccentric inventor).

The Narrative Framework: The Time-Travel Mishap
Professor Quark's time machine malfunctions, stranding the trio in a bizarre historical era or an alternate timeline. To fix the machine and return home, they must understand and apply the concepts found in the provided {{EducationMaterial}}.

How to use the {{EducationMaterial}}:
1. The Core Problem: The time machine is broken, and its repair manual has been replaced by the text in the {{EducationMaterial}}.
2. The Obstacles: Turn 3-4 key facts from the material into physical obstacles or characters they meet in this timeline.
3. The Solution: Omji must use the concepts from the material to figure out how to jump-start the time machine, while Comji causes accidental chaos.

Formatting:
- 4 distinct chapters. Include dramatic, comic-book style reactions in parentheses (e.g., *Comji sweats profusely!*).
- Target word count: ~2,000 words.

Input Material:
{EducationMaterial}""",

    """You are an expert children's author specializing in the "Korean Why? comic" style. Your task is to write a highly engaging, slapstick, and educational 2,000-word story for children aged 8-12.

Characters: Comji (impulsive/funny), Omji (smart/logical), and Professor Quark (eccentric inventor).

The Narrative Framework: The RPG Dungeon Crawl
The trio gets sucked into a virtual reality video game. To beat the game and log out, they have to navigate a multi-level dungeon where the "magic system" is entirely based on the {{EducationMaterial}}. 

How to use the {{EducationMaterial}}:
1. The Rules of the World: Treat the facts in the {{EducationMaterial}} as the absolute rules, spells, or physics of this video game world.
2. The Mini-Bosses: Each chapter should feature a puzzle or an enemy that can only be defeated by answering or understanding a concept from the material.
3. Level Up: When they correctly apply a fact, describe them "leveling up" or unlocking a new ability based on that knowledge.

Formatting:
- 4 distinct levels/chapters. Include dramatic, comic-book style reactions in parentheses.
- Target word count: ~2,000 words.

Input Material:
{EducationMaterial}""",

    """You are an expert children's author specializing in the "Korean Why? comic" style. Write a highly engaging, slapstick, and educational 2,000-word story for children aged 8-12.

Characters: Comji (impulsive/funny), Omji (smart/logical), and Professor Quark (eccentric inventor).

The Narrative Framework: The Shrink-Ray Survival
Professor Quark accidentally shrinks the team to microscopic size. A totally normal, everyday environment is now a massive, terrifying wilderness. 

How to use the {{EducationMaterial}}:
1. The Environment: Use the {{EducationMaterial}} to describe the "giant" world around them. (e.g., if the topic is biology, they are inside a giant cell; if physics, they are dodging giant atoms).
2. The Threat: A core concept from the material is threatening to crush, eat, or trap them. 
3. The Escape: They must ride, manipulate, or outsmart the elements described in the material to reach the "Enlarge" button.

Formatting:
- 4 distinct chapters. Emphasize the sheer scale of the tiny world with comedic panic.
- Target word count: ~2,000 words.

Input Material:
{EducationMaterial}""",

    """You are an expert children's author specializing in the "Korean Why? comic" style. Write a highly engaging, slapstick, and educational 2,000-word story for children aged 8-12.

Characters: Comji (impulsive/funny), Omji (smart/logical), and Professor Quark (eccentric inventor).

The Narrative Framework: The Alien Planet Expedition
The team lands on an uncharted alien planet. The planet's ecosystem, weather, or inhabitants operate strictly according to the rules of the {{EducationMaterial}}.

How to use the {{EducationMaterial}}:
1. The Discovery: The aliens or the planet's environment behave very strangely. Comji is confused and usually gets hurt in a funny way.
2. The Hypothesis: Omji reads the {{EducationMaterial}} from her scanner and realizes the planet is a literal manifestation of these facts.
3. The Alliance: They must use the knowledge from the material to help the aliens solve a massive planetary crisis before they can leave.

Formatting:
- 4 distinct chapters. Include dramatic, comic-book style reactions in parentheses.
- Target word count: ~2,000 words.

Input Material:
{EducationMaterial}""",

]
story_prompts = [
    tpl.format(EducationMaterial=EducationMaterial)
    for tpl in story_prompt_templates
]
ListOfModels = [
    
    "MiniMaxAI/MiniMax-M2.5:novita",
    "openai/gpt-oss-120b:groq",
    "meta-llama/Llama-4-Maverick-17B-128E-Instruct:sambanova"
]

import os
from openai import OpenAI

token_from_env = (
    os.getenv("HF_TOKEN")
    or os.getenv("HugginFaceToken")
    or os.getenv("HuggingFaceToken")
)

if not token_from_env:
    try:
        with open(".env", "r", encoding="utf-8") as env_file:
            for raw_line in env_file:
                line = raw_line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                key, value = line.split("=", 1)
                key = key.strip()
                value = value.strip().strip('\"').strip("'")
                if key in {"HF_TOKEN", "HugginFaceToken", "HuggingFaceToken"}:
                    token_from_env = value
                    break
    except FileNotFoundError:
        pass

if not token_from_env:
    raise ValueError("Hugging Face token not found. Set HF_TOKEN or HugginFaceToken in .env.")

client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=token_from_env,
)

In [14]:
print(story_prompts)

['You are an expert children\'s author specializing in the "Korean Why? comic" style. Your task is to write a highly engaging, slapstick, and educational 2,000-word story for children aged 8-12.\n\nCharacters: Comji (impulsive/funny), Omji (smart/logical), and Professor Quark (eccentric inventor).\n\nThe Narrative Framework: The Time-Travel Mishap\nProfessor Quark\'s time machine malfunctions, stranding the trio in a bizarre historical era or an alternate timeline. To fix the machine and return home, they must understand and apply the concepts found in the provided {EducationMaterial}.\n\nHow to use the {EducationMaterial}:\n1. The Core Problem: The time machine is broken, and its repair manual has been replaced by the text in the {EducationMaterial}.\n2. The Obstacles: Turn 3-4 key facts from the material into physical obstacles or characters they meet in this timeline.\n3. The Solution: Omji must use the concepts from the material to figure out how to jump-start the time machine, whi

In [ ]:
compare_instruction_sets(story_prompts, ListOfModels)

INSTRUCTION [1/4]
[1/3] Generating with: MiniMaxAI/MiniMax-M2.5:novita...

MODEL ANALYSIS:
API Response Time: 241.09 seconds
Word Count: 1810 words (Target: 800-950) -> Fail
Est. TTS Duration: 12m 4s
Paragraph Count: 104
Pacing: ~7.7 words per sentence
--------------------------------------------------------------------------------
FULL STORY OUTPUT:

# THE AMAZING MATTER ADVENTURE

## A Comji, Omji & Professor Quark Story

---

# CHAPTER 1: THE DISAPPEARING MANUAL

*Ka-BOOOOOM!*

Professor Quark's time machine erupted in a shower of purple sparks and glittery smoke, sputtering like an angry dragon who'd swallowed a firecracker. The machine spun wildly, tossing its three passengers against the walls like salad in a blender.

"When I said I wanted to see the future, I didn't mean like THIS!" Comji screamed, grabbing onto a safety railing that had come loose.

The machine finally stopped spinning with a triumphant *PFFFT*, like a soda can being opened. The three adventurers lay in a heap

[{'instruction_index': 1,
  'instruction_text': 'You are an expert children\'s author specializing in the "Korean Why? comic" style. Your task is to write a highly engaging, slapstick, and educational 2,000-word story for children aged 8-12.\n\nCharacters: Comji (impulsive/funny), Omji (smart/logical), and Professor Quark (eccentric inventor).\n\nThe Narrative Framework: The Time-Travel Mishap\nProfessor Quark\'s time machine malfunctions, stranding the trio in a bizarre historical era or an alternate timeline. To fix the machine and return home, they must understand and apply the concepts found in the provided {EducationMaterial}.\n\nHow to use the {EducationMaterial}:\n1. The Core Problem: The time machine is broken, and its repair manual has been replaced by the text in the {EducationMaterial}.\n2. The Obstacles: Turn 3-4 key facts from the material into physical obstacles or characters they meet in this timeline.\n3. The Solution: Omji must use the concepts from the material to fig

In [ ]:
compare_instruction_sets(story_prompts, ["meta-llama/Llama-4-Maverick-17B-128E-Instruct:sambanova"])